In [30]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import date, datetime, timedelta
# web scrapping
import bs4 as bs
import requests
import lxml
from functools import reduce
# matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from ipysigma import Sigma
from pyvis.network import Network
import requests
import re
from bs4 import BeautifulSoup
from io import StringIO
from dbconnection import MySQLDatabase
from utils import getSymbols, getData, get_last_date, get_marketid_simbols
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="pandas")
sns.set_theme()

In [2]:
db = MySQLDatabase("financialmarkets")

In [33]:
df = getSymbols('https://es.wikipedia.org/wiki/%C3%8Dndice_de_Precios_y_Cotizaciones')
df.head()

,Símbolo,Nombre,Sector,Peso en el indicador
0,AMX L[2]​,América Móvil,Servicio de Telecomunicaciones,16.63 %
1,WALMEX V[2]​,Walmex,Productos de Consumo Frecuente,7.03 %
2,FEMSA UBD[2]​,Fomento Económico Mexicano,Productos de Consumo Frecuente,11.28 %
3,TLEVISA CPO[2]​,Grupo Televisa,Servicio de Telecomunicaciones,9.08 %
4,GMEXICO B[2]​,Grupo México,Materiales,7.08 %


In [34]:
# generamos variables no existentes
df['Símbolo'] = [re.sub(r"\s+", "", x)[:-4]+'.MX' for x in df['Símbolo'].astype('str')]
df = df.rename(columns={'Símbolo':'Symbol','Nombre':'Security', 'Sector':'GICS Sector'})
df['GICS Sub-Industry'] = ['SD' for x in df['Symbol']]
df['Headquarters Location'] = ['SD' for x in df['Symbol']]
df['CIK'] = ['SD' for x in df['Symbol']]
df['Founded'] = [9999 for x in df['Symbol']]
df['Date added'] = ['0000-00-00' for x in df['Symbol']]
df = df[['Symbol','Security','GICS Sector','GICS Sub-Industry','Headquarters Location', 'Date added','CIK','Founded']]
df.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,AMXL.MX,América Móvil,Servicio de Telecomunicaciones,SD,SD,0000-00-00,SD,9999
1,WALMEXV.MX,Walmex,Productos de Consumo Frecuente,SD,SD,0000-00-00,SD,9999
2,FEMSAUBD.MX,Fomento Económico Mexicano,Productos de Consumo Frecuente,SD,SD,0000-00-00,SD,9999
3,TLEVISACPO.MX,Grupo Televisa,Servicio de Telecomunicaciones,SD,SD,0000-00-00,SD,9999
4,GMEXICOB.MX,Grupo México,Materiales,SD,SD,0000-00-00,SD,9999


**Ingresamos mercado**

In [8]:
# ---------------------------
# 3️Insertar/actualizar mercados
# ---------------------------
markets = pd.DataFrame({
    'market_name': ['IPC MX'],
    'country': ['MX'],
    'currency': ['Peso']
})
markets

,market_name,country,currency
0,IPC MX,MX,Peso


In [9]:
db.insert_to_db(markets, tabla="markets", batch_size=5000)

✅ Conexión exitosa


In [10]:
# Obtener market_id
market_id = db.execute_query("SELECT * FROM markets")
market_id

,market_id,market_name,country,currency
0,1,NASDAQ,USA,USD
1,2,S&P 500,USA,USD
2,3,IPC MX,MX,Peso


In [35]:
market_id = 3

In [36]:
df.loc[:,"market_id"] = [market_id for x in df['Symbol']]
companies = df[["market_id",'Symbol','Security','GICS Sector','GICS Sub-Industry','Date added','Headquarters Location','CIK','Founded']]
companies.columns = ["market_id",'symbol','name','sector_name','sub_industry','date_added','headquarters','cik','founded']
companies = companies.reset_index(drop=True)
companies.head()

,market_id,symbol,name,sector_name,sub_industry,date_added,headquarters,cik,founded
0,3,AMXL.MX,América Móvil,Servicio de Telecomunicaciones,SD,0000-00-00,SD,SD,9999
1,3,WALMEXV.MX,Walmex,Productos de Consumo Frecuente,SD,0000-00-00,SD,SD,9999
2,3,FEMSAUBD.MX,Fomento Económico Mexicano,Productos de Consumo Frecuente,SD,0000-00-00,SD,SD,9999
3,3,TLEVISACPO.MX,Grupo Televisa,Servicio de Telecomunicaciones,SD,0000-00-00,SD,SD,9999
4,3,GMEXICOB.MX,Grupo México,Materiales,SD,0000-00-00,SD,SD,9999


In [38]:
db.insert_to_db(companies, tabla="companies", batch_size=100)

In [39]:
db.close()

🔒 Conexión cerrada
